In [0]:
# ============================================================
# Notebook : 09_gold_market_performance
# Layer    : Gold
# Project  : Real-Time Stock Market Pipeline
#
# Purpose:
# Calculate market performance for every stock
# by comparing opening price with latest price.
# ============================================================

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
CATALOG = "realtimedata"

INPUT_TABLE = f"{CATALOG}.gold.ohlc_1min"

OUTPUT_TABLE = f"{CATALOG}.gold.market_performance"

In [0]:
ohlc_df = spark.table(INPUT_TABLE)

display(ohlc_df)

symbol,candle_time,open,high,low,close
HDFCBANK,2026-06-29T10:38:00.000Z,798.9,798.9,798.9,798.9
HDFCBANK,2026-06-30T08:40:00.000Z,798.0,798.25,797.75,797.75
HDFCBANK,2026-06-30T08:41:00.000Z,797.6,798.15,797.5,797.75
HDFCBANK,2026-06-30T08:42:00.000Z,797.85,798.25,797.7,798.05
HDFCBANK,2026-06-30T08:43:00.000Z,798.05,798.25,798.0,798.0
HDFCBANK,2026-06-30T09:48:00.000Z,798.0,798.0,797.65,797.65
HDFCBANK,2026-06-30T09:49:00.000Z,797.65,798.2,797.6,798.2
HDFCBANK,2026-06-30T09:50:00.000Z,798.1,798.45,797.95,798.1
HDFCBANK,2026-06-30T09:51:00.000Z,798.1,798.1,797.6,798.0
HDFCBANK,2026-06-30T09:52:00.000Z,798.0,799.0,797.9,798.9


In [0]:
from pyspark.sql import Window
import pyspark.sql.functions as F

# Window to identify the first candle (Opening Price)
opening_window = (
    Window
        .partitionBy("symbol")
        .orderBy(F.col("candle_time").asc())
)

# Window to identify the latest candle (Latest Price)
latest_window = (
    Window
        .partitionBy("symbol")
        .orderBy(F.col("candle_time").desc())
)

In [0]:
opening_df = (
    ohlc_df
    .withColumn(
        "row_num",
        F.row_number().over(opening_window)
    )
    .filter(F.col("row_num") == 1)
    .select(
        "symbol",
        F.col("open").alias("opening_price")
    )
)

In [0]:
latest_df = (
    ohlc_df
    .withColumn(
        "row_num",
        F.row_number().over(latest_window)
    )
    .filter(F.col("row_num") == 1)
    .select(
        "symbol",
        "candle_time",
        F.col("close").alias("latest_price")
    )
)

In [0]:
performance_df = (
    opening_df
    .join(
        latest_df,
        on="symbol",
        how="inner"
    )
)

In [0]:
performance_df = (
    performance_df
    .withColumn(
        "price_change",
        F.round(
            F.col("latest_price") - F.col("opening_price"),
            2
        )
    )
    .withColumn(
        "price_change_percent",
        F.round(
            (
                (F.col("latest_price") - F.col("opening_price"))
                / F.col("opening_price")
            ) * 100,
            2
        )
    )
)

In [0]:
performance_df = (
    performance_df
    .withColumn(
        "market_direction",
        F.when(
            F.col("price_change") > 0,
            "GAIN"
        )
        .when(
            F.col("price_change") < 0,
            "LOSS"
        )
        .otherwise("NO CHANGE")
    )
)

In [0]:
performance_df = performance_df.orderBy(
    F.col("price_change_percent").desc()
)

display(performance_df)

symbol,opening_price,candle_time,latest_price,price_change,price_change_percent,market_direction
HDFCBANK,798.9,2026-06-30T10:08:00.000Z,797.95,-0.95,-0.12,LOSS
RELIANCE,1301.0,2026-06-30T10:08:00.000Z,1293.9,-7.1,-0.55,LOSS
ICICIBANK,1387.6,2026-06-30T10:08:00.000Z,1375.2,-12.4,-0.89,LOSS
TCS,2097.9,2026-06-30T10:08:00.000Z,2031.5,-66.4,-3.17,LOSS
INFY,1036.7,2026-06-30T10:08:00.000Z,1000.4,-36.3,-3.5,LOSS


In [0]:
(
    performance_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(OUTPUT_TABLE)
)

In [0]:
gold_df = spark.table(OUTPUT_TABLE)

display(gold_df)

symbol,opening_price,candle_time,latest_price,price_change,price_change_percent,market_direction
HDFCBANK,798.9,2026-06-30T10:08:00.000Z,797.95,-0.95,-0.12,LOSS
RELIANCE,1301.0,2026-06-30T10:08:00.000Z,1293.9,-7.1,-0.55,LOSS
ICICIBANK,1387.6,2026-06-30T10:08:00.000Z,1375.2,-12.4,-0.89,LOSS
TCS,2097.9,2026-06-30T10:08:00.000Z,2031.5,-66.4,-3.17,LOSS
INFY,1036.7,2026-06-30T10:08:00.000Z,1000.4,-36.3,-3.5,LOSS


In [0]:
print("=" * 60)
print("Gold Market Performance Created Successfully")
print("=" * 60)

print(f"Total Stocks : {gold_df.count()}")

print("=" * 60)

Gold Market Performance Created Successfully
Total Stocks : 5
